<a href="https://colab.research.google.com/github/alirezzasarkar/analyze_cryptocurrency/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install yfinance keras-tuner newsapi-python transformers plotly seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 2.3 MB/s eta 0:00:00


In [ ]:
# Install required libraries (if needed in Google Colab)
# !pip install yfinance keras-tuner newsapi-python transformers

import os
import time
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import logging
import random
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

# Price prediction modeling section:
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import GRU, LSTM, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import TimeSeriesSplit
import plotly.graph_objs as go
import plotly.io as pio
import yfinance as yf
import keras_tuner as kt
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# News analysis section:
from newsapi import NewsApiClient
from transformers import pipeline

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# Logging configuration
logging.basicConfig(
    filename='model_training.log',
    level=logging.INFO,
    format='%(asctime)s:%(levelname)s:%(message)s'
)
logging.info('Model training process started')

# Default Plotly settings (for Google Colab)
pio.renderers.default = "colab"
pio.templates.default = "plotly_white"

#############################################
# ############## PRICE PREDICTION SECTION ##############
#############################################

# Define custom evaluation metrics
def mean_absolute_percentage_error_custom(y_true, y_pred):
    y_true = np.where(y_true == 0, 1e-8, y_true)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def symmetric_mean_absolute_percentage_error(y_true, y_pred):
    return np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred))) * 100

def mean_absolute_scaled_error(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred))
    mae_naive = np.mean(np.abs(y_true[1:] - y_true[:-1]))
    return mae / mae_naive

def mase(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mae_naive = mean_absolute_error(y_true[1:], y_true[:-1])
    return mae / mae_naive

# List of available cryptocurrencies
currencies = ['BTC', 'ETH', 'ADA', 'XRP', 'SOL']
print("List of available cryptocurrencies:")
for idx, currency in enumerate(currencies):
    print(f"{idx + 1}. {currency}")

# User selects a cryptocurrency
try:
    choice = int(input("Choose a cryptocurrency (number): ")) - 1
    if choice not in range(len(currencies)):
        raise ValueError
except ValueError:
    print("Invalid choice! Please enter a valid number.")
    logging.error("User made an invalid cryptocurrency selection")
    exit()

selected_currency = currencies[choice]
print(f"Selected cryptocurrency: {selected_currency}")
logging.info(f"Selected cryptocurrency: {selected_currency}")

# User selects type of analysis
print("\nSelect the type of analysis:")
print("1. Short-term (hourly data)")
print("2. Long-term (daily data)")
try:
    analysis_choice = int(input("Enter your choice (1 or 2): "))
    if analysis_choice not in [1, 2]:
        raise ValueError
except ValueError:
    print("Invalid choice! Please select either 1 (short-term) or 2 (long-term).")
    logging.error("User made an invalid selection for analysis type")
    exit()

# Settings based on type of analysis
if analysis_choice == 1:
    period = "2y"
    interval = "1h"
    sequence_size = 20
    print("You selected short-term analysis using hourly data.")
    logging.info("Analysis type: Short-term (hourly data)")
elif analysis_choice == 2:
    period = "max"
    interval = "1d"
    sequence_size = 40
    print("You selected long-term analysis using daily data.")
    logging.info("Analysis type: Long-term (daily data)")
else:
    raise ValueError("Invalid choice! Please select either 1 (short-term) or 2 (long-term).")

# Function to fetch data from Yahoo Finance
def fetch_data_yfinance(ticker, period, interval):
    try:
        data = yf.download(tickers=ticker, period=period, interval=interval)
        if not data.empty:
            data.rename(columns={
                'Open': 'open',
                'High': 'high',
                'Low': 'low',
                'Close': 'close',
                'Volume': 'volumefrom'
            }, inplace=True)
            data['volumeto'] = data['volumefrom']
            logging.info(f"Data successfully downloaded for {ticker}")
            return data
        else:
            raise Exception("Failed to fetch data from Yahoo Finance.")
    except Exception as e:
        print(f"Error fetching data: {e}")
        logging.error(f"Error downloading data: {e}")
        exit()

# Define Yahoo Finance tickers
yahoo_tickers = {
    'BTC': 'BTC-USD',
    'ETH': 'ETH-USD',
    'ADA': 'ADA-USD',
    'XRP': 'XRP-USD',
    'SOL': 'SOL-USD'
}
selected_ticker = yahoo_tickers[selected_currency]

# Fetch data
df = fetch_data_yfinance(selected_ticker, period, interval)
print(f"Fetched data: \n{df.head()}")
print(f"Last few rows of data: \n{df.tail()}")
logging.info(f"First 5 rows of data:\n{df.head()}")
logging.info(f"Last 5 rows of data:\n{df.tail()}")

# Plot candlestick chart (for visualization)
fig = go.Figure(data=[go.Candlestick(
    x=df.index,
    open=df['open'],
    high=df['high'],
    low=df['low'],
    close=df['close']
)])
fig.update_layout(
    title=f'Candlestick chart for {selected_currency}',
    xaxis_title='Date',
    yaxis_title='Price',
    xaxis_rangeslider_visible=False
)
fig.show()
logging.info("Candlestick chart displayed")

# Split data into training and testing sets
split_ratio = 0.8
split_index = int(len(df) * split_ratio)
train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()
df = pd.concat([train_df, test_df]).dropna()

# Define features and perform normalization
features = ['open', 'high', 'low', 'close']
scaler = MinMaxScaler()
scaler.fit(train_df[features])
train_scaled = scaler.transform(train_df[features])
test_scaled  = scaler.transform(test_df[features])

# Function to create time sequences
def create_multivariate_sequences(dataset, seq_size=1, target_column='close'):
    X, y = [], []
    target_index = features.index(target_column)
    for i in range(len(dataset) - seq_size):
        X.append(dataset[i:i+seq_size])
        y.append(dataset[i+seq_size, target_index])
    return np.array(X), np.array(y)

X_train, y_train = create_multivariate_sequences(train_scaled, seq_size=sequence_size, target_column='close')
X_test,  y_test  = create_multivariate_sequences(test_scaled, seq_size=sequence_size, target_column='close')

# Define the Tuner class for Walk-Forward Validation
class WalkForwardCVTuner(kt.BayesianOptimization):
    def run_trial(self, trial, X, y, batch_size=64, epochs=20, **fit_kwargs):
        tscv = TimeSeriesSplit(n_splits=5)
        val_losses = []
        for train_idx, val_idx in tscv.split(X):
            X_train_fold, X_val_fold = X[train_idx], X[val_idx]
            y_train_fold, y_val_fold = y[train_idx], y[val_idx]
            model = self.hypermodel.build(trial.hyperparameters)
            model.compile(optimizer=model.optimizer, loss=model.loss)
            early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
            history = model.fit(X_train_fold, y_train_fold,
                                validation_data=(X_val_fold, y_val_fold),
                                epochs=epochs, batch_size=batch_size,
                                callbacks=[early_stopping], verbose=0)
            val_losses.append(min(history.history['val_loss']))
        mean_val_loss = float(np.mean(val_losses))
        self.oracle.update_trial(trial.trial_id, {'val_loss': mean_val_loss})

# Define base models
def build_gru_base(hp):
    units = hp.Int('units_gru', min_value=64, max_value=512, step=64)
    dropout_rate = hp.Float('dropout_gru', 0.1, 0.5, step=0.1)
    batch_norm_choice = hp.Choice('batch_norm_gru', [True, False])
    learning_rate = hp.Choice('lr_gru', [1e-3, 1e-4, 1e-5])
    inputs = Input(shape=(sequence_size, len(features)))
    x = GRU(units, return_sequences=True)(inputs)
    if batch_norm_choice:
        x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    x = GRU(units, return_sequences=False)(x)
    if batch_norm_choice:
        x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    feature_output = Dense(50, activation='relu', name='features')(x)
    prediction = Dense(1, activation='linear', name='prediction')(feature_output)
    model = Model(inputs=inputs, outputs=prediction)
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    return model

def build_lstm_base(hp):
    units = hp.Int('units_lstm', min_value=64, max_value=512, step=64)
    dropout_rate = hp.Float('dropout_lstm', 0.1, 0.5, step=0.1)
    batch_norm_choice = hp.Choice('batch_norm_lstm', [True, False])
    learning_rate = hp.Choice('lr_lstm', [1e-3, 1e-4, 1e-5])
    inputs = Input(shape=(sequence_size, len(features)))
    x = LSTM(units, return_sequences=True)(inputs)
    if batch_norm_choice:
        x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    x = LSTM(units, return_sequences=False)(x)
    if batch_norm_choice:
        x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)
    feature_output = Dense(50, activation='relu', name='features')(x)
    prediction = Dense(1, activation='linear', name='prediction')(feature_output)
    model = Model(inputs=inputs, outputs=prediction)
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    return model

# Load saved models if available; otherwise, train new ones
if os.path.exists("saved_models/gru_base_model.h5") and os.path.exists("saved_models/lstm_base_model.h5") and os.path.exists("saved_models/meta_model.h5"):
    print("Loading saved models...")
    gru_base_model = load_model("saved_models/gru_base_model.h5")
    lstm_base_model = load_model("saved_models/lstm_base_model.h5")
    meta_model = load_model("saved_models/meta_model.h5")
    print("Saved models loaded.")
else:
    # Train GRU model using tuner
    tuner_gru = WalkForwardCVTuner(
        hypermodel=build_gru_base,
        objective='val_loss',
        max_trials=20,
        directory='tuner_dir_gru_base',
        project_name='gru_base_enhanced'
    )
    tuner_gru.search(X_train, y_train, epochs=50, batch_size=64, verbose=1)
    best_hps_gru = tuner_gru.get_best_hyperparameters(num_trials=1)[0]
    gru_base_model = tuner_gru.hypermodel.build(best_hps_gru)
    gru_base_model.compile(optimizer=Adam(learning_rate=best_hps_gru.get('lr_gru')), loss='mse')
    gru_base_model.fit(X_train, y_train, epochs=50, batch_size=64, verbose=1,
                       callbacks=[EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)])

    # Train LSTM model using tuner
    tuner_lstm = WalkForwardCVTuner(
        hypermodel=build_lstm_base,
        objective='val_loss',
        max_trials=20,
        directory='tuner_dir_lstm_base',
        project_name='lstm_base_enhanced'
    )
    tuner_lstm.search(X_train, y_train, epochs=50, batch_size=64, verbose=1)
    best_hps_lstm = tuner_lstm.get_best_hyperparameters(num_trials=1)[0]
    lstm_base_model = tuner_lstm.hypermodel.build(best_hps_lstm)
    lstm_base_model.compile(optimizer=Adam(learning_rate=best_hps_lstm.get('lr_lstm')), loss='mse')
    lstm_base_model.fit(X_train, y_train, epochs=50, batch_size=64, verbose=1,
                        callbacks=[EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)])

    # Extract features for meta model
    feature_extractor_gru = Model(inputs=gru_base_model.input, outputs=gru_base_model.get_layer('features').output)
    feature_extractor_lstm = Model(inputs=lstm_base_model.input, outputs=lstm_base_model.get_layer('features').output)
    features_gru_train = feature_extractor_gru.predict(X_train)
    features_lstm_train = feature_extractor_lstm.predict(X_train)
    X_train_meta = np.hstack((features_gru_train, features_lstm_train))

    # Build and train meta model
    def build_meta_model():
        inputs = Input(shape=(100,))
        x = Dense(64, activation='relu')(inputs)
        x = Dropout(0.3)(x)
        x = Dense(32, activation='relu')(x)
        x = Dropout(0.3)(x)
        prediction = Dense(1, activation='linear')(x)
        model = Model(inputs=inputs, outputs=prediction)
        model.compile(optimizer=Adam(1e-3), loss='mse')
        return model
    meta_model = build_meta_model()
    meta_model.fit(X_train_meta, y_train, epochs=100, batch_size=64, verbose=1,
                   callbacks=[EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)])

    # Save trained models
    if not os.path.exists("saved_models"):
        os.makedirs("saved_models")
    gru_base_model.save("saved_models/gru_base_model.h5")
    lstm_base_model.save("saved_models/lstm_base_model.h5")
    meta_model.save("saved_models/meta_model.h5")
    print("Initial models trained and saved.")

# In both cases (loaded or newly trained), extract features
feature_extractor_gru = Model(inputs=gru_base_model.input, outputs=gru_base_model.get_layer('features').output)
feature_extractor_lstm = Model(inputs=lstm_base_model.input, outputs=lstm_base_model.get_layer('features').output)

# Initial prediction to determine price recommendation
features_gru_test = feature_extractor_gru.predict(X_test)
features_lstm_test = feature_extractor_lstm.predict(X_test)
X_test_meta = np.hstack((features_gru_test, features_lstm_test))
yhat_test_meta = meta_model.predict(X_test_meta).flatten()

def inverse_scale(y_pred, scaler, features):
    y_pred_scaled = np.zeros((len(y_pred), len(features)))
    y_pred_scaled[:, features.index('close')] = y_pred
    return scaler.inverse_transform(y_pred_scaled)[:, features.index('close')]

y_test_inverse = inverse_scale(y_test, scaler, features)
yhat_test_meta_inverse = inverse_scale(yhat_test_meta, scaler, features)

# Simple price recommendation based on forecast trend:
if yhat_test_meta_inverse[-1] > yhat_test_meta_inverse[0]:
    price_recommendation = "Buy"
else:
    price_recommendation = "Sell"

#############################################
# ############## NEWS ANALYSIS SECTION ##############
#############################################

# Set up NewsAPI key and sentiment analysis pipeline
# (Note: Replace 'YOUR_NEWSAPI_KEY_HERE' with your valid API key)
newsapi = NewsApiClient(api_key='YOUR_NEWSAPI_KEY_HERE')
sentiment_pipeline = pipeline('sentiment-analysis')

# Define crypto tickers for news analysis
crypto_tickers = {
    1: 'Bitcoin',
    2: 'Ethereum',
    3: 'Binance Coin',
    4: 'Cardano',
    5: 'Solana'
}
print("Please select a cryptocurrency for news analysis:")
for key, value in crypto_tickers.items():
    print(f"{key}: {value}")
try:
    selected_crypto_index = int(input("Enter the number of the desired cryptocurrency: "))
    selected_crypto = crypto_tickers.get(selected_crypto_index)
    if selected_crypto is None:
        raise ValueError
except ValueError:
    print("Invalid selection! Please enter a valid number.")
    exit()

# Select time period for news
print("\nSelect the time period for news:")
print("1: News from the last 5 days")
print("2: News from the last 20 days")
print("3: News from the last 30 days")
try:
    selected_period = int(input("Enter the time period number: "))
except ValueError:
    print("Invalid time period!")
    exit()

if selected_period == 1:
    start_date = (datetime.now() - timedelta(days=5)).strftime('%Y-%m-%d')
elif selected_period == 2:
    start_date = (datetime.now() - timedelta(days=20)).strftime('%Y-%m-%d')
elif selected_period == 3:
    start_date = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')
else:
    print("Invalid time period. Please try again.")
    exit()

def fetch_crypto_news(ticker, start_date, page_size=10, max_pages=5):
    all_articles = []
    for page in range(1, max_pages + 1):
        response = newsapi.get_everything(
            q=ticker,
            language='en',
            from_param=start_date,
            sort_by='relevancy',
            page_size=page_size,
            page=page
        )
        articles = response.get('articles', [])
        if not articles:
            break
        all_articles.extend(articles)
        if len(articles) < page_size:
            break
    return all_articles

def process_articles(articles):
    processed = []
    for article in articles:
        processed.append({
            'title': article.get('title', ""),
            'description': article.get('description', ""),
            'url': article.get('url', ""),
            'published_at': article.get('publishedAt', "")
        })
    return processed

def get_overall_sentiment(article, threshold=0.6):
    title_sent = article.get('sentiment_title')
    title_conf = article.get('confidence_title', 0)
    desc_sent = article.get('sentiment_description')
    desc_conf = article.get('confidence_description', 0)
    if title_sent == desc_sent:
        return title_sent
    else:
        chosen = title_sent if title_conf >= desc_conf else desc_sent
        chosen_conf = title_conf if title_conf >= desc_conf else desc_conf
        return 'Neutral' if chosen_conf < threshold else chosen

def analyze_sentiment(articles):
    for article in articles:
        sentiment_title = sentiment_pipeline(article['title'])[0]
        if article['description']:
            sentiment_description = sentiment_pipeline(article['description'])[0]
        else:
            sentiment_description = {'label': 'Neutral', 'score': 0.0}
        article['sentiment_title'] = sentiment_title['label']
        article['confidence_title'] = sentiment_title['score']
        article['sentiment_description'] = sentiment_description['label']
        article['confidence_description'] = sentiment_description['score']
        article['overall_sentiment'] = get_overall_sentiment(article)
    return articles

print(f"\nFetching news related to {selected_crypto} from {start_date}...")
articles = fetch_crypto_news(selected_crypto, start_date, page_size=10, max_pages=5)
processed_articles = process_articles(articles)
analyzed_articles = analyze_sentiment(processed_articles)

positive_count = sum(1 for article in analyzed_articles if article.get('overall_sentiment') == 'POSITIVE')
negative_count = sum(1 for article in analyzed_articles if article.get('overall_sentiment') == 'NEGATIVE')
neutral_count  = sum(1 for article in analyzed_articles if article.get('overall_sentiment') == 'Neutral')

print("\nNews counts:")
print(f"Positive: {positive_count}")
print(f"Negative: {negative_count}")
print(f"Neutral: {neutral_count}")

if positive_count > negative_count and positive_count > neutral_count:
    news_recommendation = "Buy Recommendation"
elif negative_count > positive_count and negative_count > neutral_count:
    news_recommendation = "Sell Recommendation"
else:
    news_recommendation = "No Recommendation"

print(f"\nNews analysis result: {news_recommendation}")

#############################################
# ############## COMBINING RECOMMENDATIONS ##############
#############################################

def main():
    # Get price recommendation from the price prediction section
    # (variable 'price_recommendation' from the price model section)
    # Get news recommendation from the news analysis section
    print("\n--- Final Results ---")
    print(f"Price Recommendation: {price_recommendation}")
    print(f"News Recommendation: {news_recommendation}")

    # Simple combination logic:
    if price_recommendation == "Buy" and news_recommendation in ["Buy Recommendation", "Buy"]:
        final_recommendation = "Strong Buy Recommendation"
    elif price_recommendation == "Sell" and news_recommendation in ["Sell Recommendation", "Sell"]:
        final_recommendation = "Strong Sell Recommendation"
    else:
        final_recommendation = "Final decision requires further review"

    print("\nFinal Recommendation:", final_recommendation)
    logging.info("Final Recommendation: " + final_recommendation)

if __name__ == "__main__":
    main()

#############################################
# ############## ONLINE TRAINING UPDATE ##############
#############################################

def online_training_update(new_data):
    """
    This function receives new data, preprocesses it, and updates the base models and meta model online.
    """
    new_data = new_data[features].dropna()
    if new_data.empty:
        print("No new data available for online update.")
        return
    new_data_scaled = scaler.transform(new_data)
    X_new, y_new = create_multivariate_sequences(new_data_scaled, seq_size=sequence_size, target_column='close')
    if len(X_new) == 0:
        print("Not enough new data to form sequences.")
        return
    print("Performing online training update with new data...")
    gru_base_model.fit(X_new, y_new, epochs=5, batch_size=64, verbose=1)
    lstm_base_model.fit(X_new, y_new, epochs=5, batch_size=64, verbose=1)
    features_gru_new = feature_extractor_gru.predict(X_new)
    features_lstm_new = feature_extractor_lstm.predict(X_new)
    X_new_meta = np.hstack((features_gru_new, features_lstm_new))
    meta_model.fit(X_new_meta, y_new, epochs=5, batch_size=64, verbose=1)
    # Save models again after online update
    gru_base_model.save("saved_models/gru_base_model.h5")
    lstm_base_model.save("saved_models/lstm_base_model.h5")
    meta_model.save("saved_models/meta_model.h5")
    print("Online training update completed.")

print("\nStarting online training update loop...")
while True:
    print("Fetching new data for online update...")
    if analysis_choice == 1:
        new_data = fetch_data_yfinance(selected_ticker, period="1d", interval="1h")
        update_interval = 3600  # 1 hour
    else:
        new_data = fetch_data_yfinance(selected_ticker, period="7d", interval="1d")
        update_interval = 86400  # 1 day
    online_training_update(new_data)
    print("Waiting for next online update...")
    time.sleep(update_interval)

Trial 7 Complete [00h 02m 11s]
val_loss: 0.005494708308833651

Best val_loss So Far: 4.1595206403144404e-05
Total elapsed time: 00h 22m 15s

Search: Running Trial #8

Value             |Best Value So Far |Hyperparameter
128               |448               |units_gru
0.2               |0.2               |dropout_gru
1                 |0                 |batch_norm_gru
1e-05             |0.001             |lr_gru

